# Stanford MRNet Pre-Training: Knee MRI Foundation Backbone (Auto-Discovery)

**Stage 1 Medical Foundation Training:**
- **Dataset**: Stanford MRNet (1,370 fully labeled knee MRI exams).
- **Anatomical Views**: Sagittal, Coronal, and Axial series.
- **Pathologies**: ACL Tear, Meniscus Tear, General Abnormality.
- **Robust Auto-Discovery**: Automatically adapts to any Kaggle MRNet dataset mirror/folder structure.

In [ ]:
import os, re, gc, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import timm
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Configuration & Robust Dataset Auto-Discovery

In [ ]:
class CFG:
    seed           = 42
    epochs         = 10
    batch_size     = 4
    accum_steps    = 4
    lr             = 3e-4
    lr_min         = 1e-6
    weight_decay   = 1e-2
    backbone       = 'efficientnet_b2'
    slices_per_view= 12
    image_size     = (256, 256)
    num_classes    = 3
    tasks          = ['abnormal', 'acl', 'meniscus']
    device         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    use_amp        = True
    num_workers    = 4

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(CFG.seed)

# Print all mounted inputs in Kaggle
print('Mounted inputs in /kaggle/input:')
if Path('/kaggle/input').exists():
    for item in Path('/kaggle/input').iterdir():
        print(f'  - {item.name}')

# Robust MRNet dataset locator
MRNET_DIR = None
acl_csv = None

# 1. Search for any ACL label file (hyphen or underscore)
for pat in ['*train*acl*.csv', '*acl*.csv', '*train*abnormal*.csv']:
    matches = list(Path('/kaggle/input').rglob(pat))
    if matches:
        acl_csv = matches[0]
        MRNET_DIR = acl_csv.parent
        break

# 2. Fallback: search for sagittal folder
if MRNET_DIR is None:
    sag_folders = [d for d in Path('/kaggle/input').rglob('*sagittal*') if d.is_dir()]
    if sag_folders:
        MRNET_DIR = sag_folders[0].parent.parent

print(f'Discovered MRNet Root: {MRNET_DIR}')
if MRNET_DIR is not None:
    print('Contents of MRNet root:')
    for p in list(MRNET_DIR.iterdir())[:10]:
        print(f'  - {p.name}')
else:
    raise FileNotFoundError(
        'MRNet dataset not found! Please make sure you clicked "+ Add Input" in the right panel '
        'and added the dataset (e.g. "abdurrahimsheikh/mrnet-dataset" or search "mrnet").'
    )

## 2. Dynamic Label Parser & NumPy Volumetric Dataset

In [ ]:
def find_csv(split, task):
    for pat in [
        f'{split}-{task}.csv',
        f'{split}_{task}.csv',
        f'*{split}*{task}*.csv',
    ]:
        m = list(MRNET_DIR.rglob(pat))
        if m: return m[0]
    raise FileNotFoundError(f'Could not find {split} {task} label CSV in {MRNET_DIR}')

def load_labels(split='train'):
    dfs = {}
    for task in CFG.tasks:
        csv_path = find_csv(split, task)
        # Handle both header and headerless CSVs
        df_tmp = pd.read_csv(csv_path)
        if df_tmp.shape[1] == 2 and not np.issubdtype(df_tmp.iloc[0, 0], np.number):
            # Headerless
            df = pd.read_csv(csv_path, header=None, names=['case', task], dtype={'case': str})
        else:
            df = pd.read_csv(csv_path, header=None, names=['case', task], dtype={'case': str})
            if df.iloc[0]['case'].lower() in ['case', 'id', 'case_id', 'study']:
                df = df.iloc[1:].reset_index(drop=True)
        df['case'] = df['case'].astype(str).str.zfill(4)
        df[task] = df[task].astype(np.float32)
        dfs[task] = df
    merged = dfs['abnormal'].merge(dfs['acl'], on='case').merge(dfs['meniscus'], on='case')
    return merged

train_df = load_labels('train')
valid_df = load_labels('valid')
print(f'Train cases: {len(train_df)} | Valid cases: {len(valid_df)}')
print('Class balance (Train):')
print(train_df[['abnormal', 'acl', 'meniscus']].mean())

# Auto-locate plane subdirectories
def find_plane_dir(split, plane):
    for d in MRNET_DIR.rglob(f'*{plane}*'):
        if d.is_dir() and (split in str(d).lower() or split in d.parent.name.lower()):
            return d
    # Fallback
    d = MRNET_DIR / split / plane
    return d

PLANE_DIRS = {
    'train': {p: find_plane_dir('train', p) for p in ['sagittal', 'coronal', 'axial']},
    'valid': {p: find_plane_dir('valid', p) for p in ['sagittal', 'coronal', 'axial']},
}
print('Plane directories (Train):', PLANE_DIRS['train'])

def sample_slices_npy(arr_3d, n_slices):
    S = arr_3d.shape[0]
    if S >= n_slices:
        idx = np.linspace(0, S - 1, n_slices, dtype=int)
    else:
        idx = list(range(S)) + [S - 1] * (n_slices - S)
    return arr_3d[idx]

class MRNetDataset(Dataset):
    def __init__(self, df, split='train', transform=None):
        self.df = df.reset_index(drop=True)
        self.split = split
        self.tf = transform

    def __len__(self): return len(self.df)

    def _load_plane(self, case, plane):
        p_dir = PLANE_DIRS[self.split][plane]
        npy_path = p_dir / f'{case}.npy'
        if not npy_path.exists():
            # Try unpadded case ID (e.g. 1 instead of 0001)
            alt_path = p_dir / f'{int(case)}.npy'
            if alt_path.exists(): npy_path = alt_path
        if npy_path.exists():
            arr = np.load(str(npy_path))
            arr = arr.astype(np.float32)
            mi, ma = arr.min(), arr.max()
            if ma > mi: arr = (arr - mi) / (ma - mi) * 255.0
            arr = arr.astype(np.uint8)
        else:
            arr = np.zeros((CFG.slices_per_view, 256, 256), dtype=np.uint8)
        return sample_slices_npy(arr, CFG.slices_per_view)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        case = row['case']

        sag = self._load_plane(case, 'sagittal')
        cor = self._load_plane(case, 'coronal')
        axi = self._load_plane(case, 'axial')

        all_slices = np.concatenate([sag, cor, axi], axis=0) # (36, 256, 256)

        imgs = []
        for s in all_slices:
            im = Image.fromarray(s, mode='L').convert('RGB')
            imgs.append(self.tf(im))
        images = torch.stack(imgs) # (36, 3, 256, 256)

        labels = torch.tensor(row[CFG.tasks].values.astype(np.float32))
        return {'images': images, 'labels': labels, 'case': case}

train_transform = T.Compose([
    T.Resize(CFG.image_size),
    T.RandomHorizontalFlip(0.5),
    T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    T.ColorJitter(brightness=0.08, contrast=0.08),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_transform = T.Compose([
    T.Resize(CFG.image_size),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
print('MRNet DataLoaders ready.')

## 3. Architecture: Dynamic Attention-Pool EfficientNet-B2

In [ ]:
class AttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(nn.Linear(dim, 128), nn.Tanh(), nn.Linear(128, 1))

    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=1)
        return (x * w).sum(dim=1)

class MRNetKneeModel(nn.Module):
    def __init__(self, backbone_name='efficientnet_b2', pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        D = self.backbone.num_features # 1408
        self.pool = AttentionPool(D)
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(D, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(512, CFG.num_classes),
        )

    def forward(self, images):
        B, S, C, H, W = images.shape
        feats = self.backbone(images.view(B * S, C, H, W)).view(B, S, -1)
        pooled = self.pool(feats)
        return self.head(pooled)

model = MRNetKneeModel(CFG.backbone, pretrained=True).to(CFG.device)
print(f'Model initialized: {CFG.backbone} ({sum(p.numel() for p in model.parameters()):,} params)')

## 4. Foundation Pretraining Loop (10 Epochs)

In [ ]:
train_ds = MRNetDataset(train_df, 'train', train_transform)
valid_ds = MRNetDataset(valid_df, 'valid', val_transform)

train_ld = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True,
                      num_workers=CFG.num_workers, pin_memory=True, drop_last=True)
valid_ld = DataLoader(valid_ds, batch_size=CFG.batch_size, shuffle=False,
                      num_workers=CFG.num_workers, pin_memory=True)

pos_counts = train_df[CFG.tasks].sum().values
pos_weights = torch.tensor((len(train_df) - pos_counts) / pos_counts, dtype=torch.float32).to(CFG.device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
total_steps = CFG.epochs * len(train_ld) // CFG.accum_steps
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(total_steps, 1), eta_min=CFG.lr_min)
scaler = torch.cuda.amp.GradScaler(enabled=CFG.use_amp)

best_auc = 0.0
best_state = None

for epoch in range(CFG.epochs):
    model.train()
    running_loss = 0.0; steps = 0
    optimizer.zero_grad()
    
    for i, batch in enumerate(tqdm(train_ld, desc=f'Epoch {epoch+1} Train')):
        imgs = batch['images'].to(CFG.device, non_blocking=True)
        lbls = batch['labels'].to(CFG.device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=CFG.use_amp):
            logits = model(imgs)
            loss = criterion(logits, lbls) / CFG.accum_steps
        scaler.scale(loss).backward()
        running_loss += loss.item() * CFG.accum_steps; steps += 1
        if (i + 1) % CFG.accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(); scheduler.step()

    # Validation
    model.eval()
    all_p, all_y = [], []
    with torch.no_grad():
        for batch in tqdm(valid_ld, desc=f'Epoch {epoch+1} Val'):
            imgs = batch['images'].to(CFG.device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=CFG.use_amp):
                preds = torch.sigmoid(model(imgs))
            all_p.append(preds.cpu().numpy())
            all_y.append(batch['labels'].cpu().numpy())
    
    preds_all = np.concatenate(all_p)
    labels_all = np.concatenate(all_y)
    
    task_aucs = {}
    for t_i, task in enumerate(CFG.tasks):
        task_aucs[task] = roc_auc_score(labels_all[:, t_i], preds_all[:, t_i])
    mean_auc = np.mean(list(task_aucs.values()))
    
    print(f'\nEpoch {epoch+1}/{CFG.epochs} - Loss: {running_loss/max(steps,1):.4f} - Mean AUC: {mean_auc:.4f}')
    for task, auc in task_aucs.items():
        print(f'  {task.upper():12s}: AUC = {auc:.4f}')
    
    if mean_auc > best_auc:
        best_auc = mean_auc
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'  >>> Best Model Saved (AUC={best_auc:.4f})!')

# Save final pre-trained weights
output_path = 'efficientnet_b2_mrnet.pth'
torch.save(best_state, output_path)
print(f'\nFoundation Pre-Training Complete!')
print(f'Saved Knee Foundation Weights to: {output_path}')
print(f'Best Validation AUC on Stanford MRNet: {best_auc:.4f}')